In [ ]:
# ros2 launch dsr_bringup2 dsr_bringup2_rviz.launch.py mode:=real host:=110.120.1.68 model:=e0509
# ros2 run dsr_gripper gripper_service

In [ ]:
import rclpy
import DR_init
from DSR_ROBOT2 import (
    posx, posj, movel, movej, wait, DR_BASE,
    set_robot_mode, get_robot_mode, get_current_posx,
    ROBOT_MODE_MANUAL, ROBOT_MODE_AUTONOMOUS,
)
from dsr_gripper import gripper_open, gripper_close, gripper_cmd

import cv2
import json
import time
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from enum import Enum, auto

ROBOT_ID = "dsr01"
ROBOT_MODEL = "e0509"
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

rclpy.init()
node = rclpy.create_node("doosan_teaching_node", namespace=ROBOT_ID)
DR_init.__dsr__node = node

print("초기화 완료. 티칭 단계를 진행하세요.")

## 1. 티칭 - manual 조작으로 pick 위치 저장

In [ ]:
set_robot_mode(ROBOT_MODE_MANUAL)
wait(1)
print(f"현재 로봇 모드: {get_robot_mode()} (0이면 manual 성공)")
print("직접교시 버튼을 누르고 Pick 위치로 팔을 이동시킨 후 다음 셀을 실행하세요.")

In [ ]:
current_x, _ = get_current_posx()
PICK_POS = [round(v, 3) for v in current_x]
print("[Pick 위치 저장 완료]")
print("PICK_POS =", PICK_POS)

## 2. 디버그를 위한 검증 시퀀스 - pos, movel 등 검증

In [ ]:
SPEED_L, ACC_L = 60, 60  # mm/s, 안전 기동 속도

PICK_1 = [330.492, 185.379, 524.939, 154.139, -178.609, 66.843]
PICK_2 = [330.492, 185.379, 209.479, 154.139, -178.609, 66.843]  # Z만 변경
PICK_3 = [324.159, -208.944, 524.939, 154.139, -178.609, 66.843]
PICK_4 = [324.159, -208.944, 199.813, 154.139, -178.609, 66.843]

print("4점 좌표 세팅 완료.")

In [ ]:
try:
    START_POS, _ = get_current_posx()  # 시작 전 현재 위치 기록
    START_POS = [round(v, 3) for v in START_POS]
    print("시작 위치 기록:", START_POS)

    gripper_open(); wait(1)
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)

    movel(posx(PICK_2), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE); wait(1)
    gripper_close(current=50); wait(3)

    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)
    movel(posx(PICK_4), vel=SPEED_L * 0.5, acc=ACC_L * 0.5, ref=DR_BASE); wait(1)

    gripper_open(); wait(3)
    movel(posx(PICK_3), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)
    movel(posx(PICK_1), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)
    print("4점 시퀀스 완료")
    movel(posx(START_POS), vel=SPEED_L, acc=ACC_L, ref=DR_BASE); wait(1)  # 시작 위치로 복귀
    print("시작 위치로 복귀")
except Exception as e:
    print(f"구동 중 에러 발생: {e}")

## 3. RealSense 카메라 연동 - color_image / depth_frame 얻기
아래 비전 함수(`get_object_contours`, `get_object_width` 등)들이 입력으로 필요로 하는 `color_image`, `depth_frame`, `intrinsics`를 여기서 만듭니다.
`pyrealsense2` 패키지가 로봇 PC에 설치되어 있어야 합니다.

In [ ]:
import pyrealsense2 as rs

pipeline = rs.pipeline()
rs_config = rs.config()
rs_config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
rs_config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
pipeline.start(rs_config)
align = rs.align(rs.stream.color)  # depth를 컬러 이미지 좌표에 맞춰 정렬

def get_frame():
    """RealSense에서 정렬된 컬러 이미지, depth 프레임, 카메라 intrinsics를 한 번에 얻는다."""
    frames = pipeline.wait_for_frames()
    aligned = align.process(frames)
    color_frame = aligned.get_color_frame()
    depth_frame = aligned.get_depth_frame()
    color_image = np.asanyarray(color_frame.get_data())
    intrinsics = color_frame.profile.as_video_stream_profile().intrinsics
    return color_image, depth_frame, intrinsics

print("RealSense 파이프라인 시작됨. get_frame()으로 언제든 프레임을 받아올 수 있습니다.")

## 3-1. 핸드-아이 캘리브레이션 - 카메라 좌표계 ↔ 로봇 좌표계
비전이 찾아낸 접촉점은 "카메라가 본 좌표"이고, 로봇은 "로봇 base 좌표"만 이해합니다.
이 둘을 연결하려면 같은 지점을 두 좌표계에서 각각 측정한 대응쌍이 **3쌍 이상** 필요합니다.

**순서**
1. `pixel_depth_to_camera_point()`로 픽셀+depth → 카메라 3D 좌표(mm) 변환
2. Manual 모드로 로봇을 어떤 지점에 직접 갖다 댐 → `get_current_posx()`로 로봇 좌표 기록
3. 동시에 카메라로 그 지점을 찍고, 이미지에서 그 지점을 클릭 → 카메라 3D 좌표 계산
4. 2~3을 지점을 바꿔가며 3번 이상 반복 → `estimate_hand_eye_transform()`으로 (R, t) 추정
5. 이후로는 `camera_to_robot(cam_point)`로 아무 카메라 좌표나 로봇 좌표로 변환 가능


In [ ]:
def pixel_depth_to_camera_point(x_px: float, y_px: float, depth_frame, intrinsics):
    """픽셀 좌표(x_px, y_px)와 depth_frame으로 카메라 기준 3D 점(mm)을 계산.
    rs2_deproject_pixel_to_point는 렌즈 왜곡까지 반영해서 역투영해준다.
    해당 픽셀의 depth가 0(측정 실패)이면 None을 반환한다."""
    depth_m = depth_frame.get_distance(int(round(x_px)), int(round(y_px)))
    if depth_m <= 0:
        return None
    point_m = rs.rs2_deproject_pixel_to_point(intrinsics, [float(x_px), float(y_px)], depth_m)
    return np.array(point_m) * 1000.0  # m -> mm

print("pixel_depth_to_camera_point() 준비 완료.")

In [ ]:
import matplotlib.pyplot as plt

CALIB_PATH = Path("hand_eye_calib.json")
calib_pairs = []  # [(cam_point_mm(xyz), robot_point_mm(xyz)), ...]

def load_calib_pairs():
    global calib_pairs
    if CALIB_PATH.exists():
        with open(CALIB_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        calib_pairs = [(np.array(p["cam"]), np.array(p["robot"])) for p in data]
        print(f"저장된 캘리브레이션 대응점 {len(calib_pairs)}개 로드됨")
    else:
        print("저장된 대응점 없음 - 새로 수집하세요")

def save_calib_pairs():
    data = [{"cam": c.tolist(), "robot": r.tolist()} for c, r in calib_pairs]
    with open(CALIB_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def collect_calibration_point():
    """1) manual 모드에서 로봇 TCP를 목표 지점에 직접 갖다 댄 뒤 이 함수를 실행하세요.
    2) 현재 로봇 좌표를 기록하고 카메라 프레임을 띄웁니다.
    3) 뜬 이미지에서 로봇 TCP가 닿은 그 지점을 클릭하면 대응쌍 1개가 추가됩니다."""
    robot_xyz, _ = get_current_posx()
    robot_xyz = np.array(robot_xyz[:3], dtype=float)  # mm, base 좌표계

    color_image, depth_frame, intrinsics = get_frame()

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB))
    plt.title("로봇 TCP가 닿은 지점을 클릭하세요")
    clicked = plt.ginput(1, timeout=0)
    plt.close()

    if not clicked:
        print("클릭이 없어 취소되었습니다.")
        return

    px, py = clicked[0]
    cam_xyz = pixel_depth_to_camera_point(px, py, depth_frame, intrinsics)
    if cam_xyz is None:
        print("해당 픽셀에서 depth를 읽지 못했습니다(0). 다시 시도하세요.")
        return

    calib_pairs.append((cam_xyz, robot_xyz))
    save_calib_pairs()
    print(f"[{len(calib_pairs)}] 카메라좌표(mm)={cam_xyz.round(1)}  로봇좌표(mm)={robot_xyz.round(1)}  (저장됨)")

load_calib_pairs()

In [ ]:
HAND_EYE_R = None  # (3,3) 회전행렬
HAND_EYE_T = None  # (3,) 이동벡터,  robot_xyz ≈ HAND_EYE_R @ cam_xyz + HAND_EYE_T

def estimate_hand_eye_transform(pairs=None):
    """대응점 쌍(카메라 3D, 로봇 3D)으로 강체변환(R, t)을 Kabsch(SVD) 방법으로 추정.
    fit_physics_constant()가 (width, force) 쌍으로 계수 하나를 구했던 것과 같은 원리로,
    여기서는 (cam_xyz, robot_xyz) 쌍으로 회전+이동을 구한다. 최소 3개(비공선) 필요."""
    global HAND_EYE_R, HAND_EYE_T
    pairs = pairs if pairs is not None else calib_pairs
    if len(pairs) < 3:
        print(f"대응점이 {len(pairs)}개뿐입니다. 최소 3개(비공선) 필요합니다.")
        return None, None

    cam_pts = np.array([p[0] for p in pairs])
    robot_pts = np.array([p[1] for p in pairs])

    cam_centroid = cam_pts.mean(axis=0)
    robot_centroid = robot_pts.mean(axis=0)
    cam_c = cam_pts - cam_centroid
    robot_c = robot_pts - robot_centroid

    H = cam_c.T @ robot_c
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:  # 반사(reflection) 방지
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = robot_centroid - R @ cam_centroid

    residuals = [np.linalg.norm(R @ c + t - r) for c, r in zip(cam_pts, robot_pts)]
    print(f"캘리브레이션 완료 (점 {len(pairs)}개). 평균 오차={np.mean(residuals):.2f}mm, 최대 오차={np.max(residuals):.2f}mm")

    HAND_EYE_R, HAND_EYE_T = R, t
    return R, t

def camera_to_robot(cam_point_mm):
    """카메라 3D 좌표(mm) 1개를 로봇 base 좌표(mm)로 변환."""
    if HAND_EYE_R is None:
        raise RuntimeError("핸드-아이 캘리브레이션이 안 되어 있습니다. estimate_hand_eye_transform()를 먼저 실행하세요.")
    return HAND_EYE_R @ np.array(cam_point_mm) + HAND_EYE_T

# 대응점이 이미 3개 이상 쌓여 있다면 바로 추정, 아니면 collect_calibration_point()로 더 모으세요.
if len(calib_pairs) >= 3:
    HAND_EYE_R, HAND_EYE_T = estimate_hand_eye_transform()
else:
    print(f"현재 대응점 {len(calib_pairs)}개. collect_calibration_point()를 3번 이상 실행해 모으세요.")

**참고**
- 대응점은 로봇 작업공간 전체에 퍼지도록(같은 평면 위 3점이 아니라 높이도 다르게) 잡을수록 정확도가 올라갑니다.
- 오차(위 출력의 평균/최대 mm)가 크면 대응점을 더 추가하고 `estimate_hand_eye_transform()`을 다시 실행하세요.
- 이 변환은 위치(x,y,z)만 다룹니다. 그리퍼가 접근하는 회전(자세)은 지금처럼 `approach_pos`의 자세값을 그대로 씁니다.

## 4. 비전 함수 - 폭/각도/법선 추출

In [ ]:
def get_object_contours(color_image: np.ndarray):
    """컬러 이미지에서 물체 윤곽선을 추출한다."""
    gray = cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours


def get_object_width(contour: np.ndarray, depth_frame, intrinsics, fallback_px_to_mm: float = 1.0) -> float:
    """윤곽선의 최소외접사각형 폭(픽셀)을 depth로 실제 mm 폭으로 변환.
    depth_frame/intrinsics가 없으면(2D fallback) 픽셀값에 임의 상수를 곱해 근사한다."""
    rect = cv2.minAreaRect(contour)
    (cx, cy), (w_px, h_px), angle = rect
    w_px = min(w_px, h_px)

    if depth_frame is None or intrinsics is None:
        return w_px * fallback_px_to_mm

    depth = depth_frame.get_distance(int(cx), int(cy))  # meter
    w_real_m = w_px * depth / intrinsics.fx
    return w_real_m * 1000.0  # mm


def get_approach_angle(contour: np.ndarray) -> float:
    """윤곽선의 주축 방향(그리퍼 접근각 θ, degree)을 반환."""
    rect = cv2.minAreaRect(contour)
    return rect[-1]


def get_surface_normals(depth_frame, contact_points_px, window: int = 3):
    """접촉점 주변 depth 차분으로 3D 법선벡터를 근사 계산."""
    normals = []
    for (x, y) in contact_points_px:
        dzdx = depth_frame.get_distance(x + window, y) - depth_frame.get_distance(x - window, y)
        dzdy = depth_frame.get_distance(x, y + window) - depth_frame.get_distance(x, y - window)
        n = np.array([-dzdx, -dzdy, 1.0])
        normals.append(n / (np.linalg.norm(n) + 1e-9))
    return normals


def estimate_mass_area_based(width_mm: float, k_area: float = 1e-5) -> float:
    """깊이 정보가 없을 때: 면적 기반 질량 근사 (kg).
    M ≈ area_ratio * w^2  (grasp_theory.md 2절) - 폭이 커지면 면적은 제곱으로 늘어나므로 w^2에 비례.
    k_area(=area_ratio)는 플레이스홀더 상수이며, 1단계 실측(폭/무게/안 미끄러지는 최소 current)으로
    반드시 재보정해야 한다. 실측 데이터가 쌓이면 fit_physics_constant()가 이 근사의 오차까지
    포함해서 F = C * mass 의 C를 다시 맞춰준다."""
    return (width_mm ** 2) * k_area


def get_object_contact_points(contour: np.ndarray):
    """최소외접사각형의 마주보는 두 변 중점을 접촉점 후보로 사용 (단순 버전)."""
    box = cv2.boxPoints(cv2.minAreaRect(contour))
    p1, p2 = box[0], box[2]
    return p1, p2

## 5. 파지 계산 - GraspPlanner (LUT 기반 naive 모드 + 물리식 기반 physics 모드)

In [ ]:
@dataclass
class GraspCandidate:
    contact_points: tuple  # ((x1,y1,z1), (x2,y2,z2))
    normals: tuple          # (n1, n2)
    width_mm: float
    center_of_mass: np.ndarray


class GraspPlanner:
    def __init__(self, mu: float = 0.5, safety_k: float = 1.5, gravity: float = 9.81):
        self.mu = mu
        self.k = safety_k
        self.g = gravity
        self.lut = {}  # width(mm, 반올림) -> 실측 최적 힘(N)
        self.fitted_C = None  # accept된 실측 데이터로 회귀한 F = C * mass 계수 (fitted 모드용)

    def add_calibration(self, width: float, force: float):
        self.lut[round(width, 1)] = force

    def _naive_force(self, width: float, default: float = 20.0) -> float:
        return self.lut.get(round(width, 1), default)

    def _physics_force(self, mass_kg: float, f_external: float = 0.0) -> float:
        # 2 * F_grasp * mu >= M*g + F_external
        return (self.k * (mass_kg * self.g) + f_external) / (2 * self.mu)

    def calculate_force(self, width: float, mass_kg: float, f_external: float = 0.0, mode: str = "physics") -> float:
        if mode == "naive":
            return self._naive_force(width)
        if mode == "fitted":
            if self.fitted_C is None:
                print("[GraspPlanner] fitted_C가 아직 없어 physics 기본값으로 대체합니다. fit_physics_constant()를 먼저 실행하세요.")
                return self._physics_force(mass_kg, f_external)
            return self.fitted_C * mass_kg + f_external
        return self._physics_force(mass_kg, f_external)

    def stability_score(self, candidate: GraspCandidate) -> float:
        n1, n2 = candidate.normals
        parallelism = -float(np.dot(n1, n2))
        mid_point = (np.array(candidate.contact_points[0]) + np.array(candidate.contact_points[1])) / 2
        dist_to_com = float(np.linalg.norm(mid_point - candidate.center_of_mass))
        com_alignment = 1.0 / (1.0 + dist_to_com)
        return parallelism + com_alignment

    def is_force_closure(self, candidate: GraspCandidate, friction_tol: float = 0.2) -> bool:
        n1, n2 = candidate.normals
        antipodal = np.linalg.norm(n1 + n2) < friction_tol
        alpha = np.arccos(np.clip(np.dot(n1, -n2), -1.0, 1.0))
        within_friction_cone = alpha <= np.arctan(self.mu)
        return antipodal and within_friction_cone

    def select_best_grasp(self, candidates: list):
        valid = [c for c in candidates if self.is_force_closure(c)]
        if not valid:
            return None
        return max(valid, key=self.stability_score)

## 6. 상태머신 - GraspController (실제 로봇 이동/그리퍼 함수는 주입받음)

In [ ]:
class GraspState(Enum):
    IDLE = auto()
    APPROACH = auto()
    DESCEND = auto()
    GRASP = auto()
    HOLD = auto()
    LIFT = auto()
    RELEASE = auto()
    DONE = auto()
    ERROR = auto()


class GraspController:
    def __init__(self, move_fn, gripper_close_fn, gripper_open_fn, get_force_sensor_fn=None):
        self.state = GraspState.IDLE
        self.move_fn = move_fn
        self.gripper_close_fn = gripper_close_fn
        self.gripper_open_fn = gripper_open_fn
        self.get_force_sensor_fn = get_force_sensor_fn
        self.weight_offset = 0.0

    def calibrate_gravity_offset(self):
        if self.get_force_sensor_fn:
            self.weight_offset = self.get_force_sensor_fn()

    def apply_gravity_offset(self) -> float:
        if not self.get_force_sensor_fn:
            return 0.0
        return self.get_force_sensor_fn() - self.weight_offset

    def execute_grasp(self, approach_pos, pick_pos, target_force: float,
                       hold_duration: float = 1.0, poll_interval: float = 0.1):
        try:
            self.state = GraspState.APPROACH
            self.move_fn(approach_pos)

            self.state = GraspState.DESCEND
            self.move_fn(pick_pos)

            self.state = GraspState.GRASP
            self.gripper_close_fn(current=target_force)

            self.state = GraspState.HOLD
            elapsed = 0.0
            while elapsed < hold_duration:
                net_force = self.apply_gravity_offset()
                # TODO: net_force가 임계치를 넘으면(미끄러짐 감지) target_force를 올리는 보정 로직 추가 가능
                time.sleep(poll_interval)
                elapsed += poll_interval

            self.state = GraspState.LIFT
            self.move_fn(approach_pos)

            self.state = GraspState.DONE
            return True

        except Exception as e:
            self.state = GraspState.ERROR
            print(f"[GraspController] 에러 발생: {e}")
            return False

    def release_at(self, place_pos):
        self.state = GraspState.RELEASE
        self.move_fn(place_pos)
        self.gripper_open_fn()
        self.state = GraspState.DONE

## 7. RobotSystem - 비전+플래너+컨트롤러 결합
> ⚠️ 이전 버전에 `np := ...` 월러스 연산자로 `numpy` 모듈 별칭 `np`를 덮어쓰는 버그가 있었습니다. 아래 코드는 수정된 버전입니다.

In [ ]:
class RobotSystem:
    def __init__(self, move_fn, gripper_close_fn, gripper_open_fn,
                 get_force_sensor_fn=None, mu: float = 0.5, safety_k: float = 1.5):
        self.planner = GraspPlanner(mu=mu, safety_k=safety_k)
        self.controller = GraspController(
            move_fn=move_fn,
            gripper_close_fn=gripper_close_fn,
            gripper_open_fn=gripper_open_fn,
            get_force_sensor_fn=get_force_sensor_fn,
        )

    def build_candidates(self, color_image, depth_frame, intrinsics=None):
        contours = get_object_contours(color_image)
        candidates = []
        for c in contours:
            width_mm = get_object_width(c, depth_frame, intrinsics)
            com_xy = c.reshape(-1, 2).mean(axis=0)
            com = np.array([com_xy[0], com_xy[1], 0.0])

            p1, p2 = get_object_contact_points(c)  # 버그 수정: cv2 직접 호출, np 안 건드림
            normals = (get_surface_normals(depth_frame, [tuple(p1), tuple(p2)])
                       if depth_frame is not None else
                       [np.array([1.0, 0, 0]), np.array([-1.0, 0, 0])])

            # 접촉점을 픽셀이 아니라 "카메라 기준 3D 좌표(mm)"로 저장한다.
            # depth가 없거나 읽기 실패하면 픽셀+z=0으로 대체(2D fallback, 로봇 좌표 변환 불가).
            if depth_frame is not None and intrinsics is not None:
                cam_p1 = pixel_depth_to_camera_point(p1[0], p1[1], depth_frame, intrinsics)
                cam_p2 = pixel_depth_to_camera_point(p2[0], p2[1], depth_frame, intrinsics)
                cam_p1 = cam_p1 if cam_p1 is not None else np.array([p1[0], p1[1], 0.0])
                cam_p2 = cam_p2 if cam_p2 is not None else np.array([p2[0], p2[1], 0.0])
            else:
                cam_p1 = np.array([p1[0], p1[1], 0.0])
                cam_p2 = np.array([p2[0], p2[1], 0.0])

            candidates.append(GraspCandidate(
                contact_points=(tuple(cam_p1), tuple(cam_p2)),
                normals=tuple(normals),
                width_mm=width_mm,
                center_of_mass=com,
            ))
        return candidates

    def run_task(self, color_image, depth_frame, approach_pos, mode: str = "physics",
                 intrinsics=None, f_external: float = 0.0):
        candidates = self.build_candidates(color_image, depth_frame, intrinsics)
        best = self.planner.select_best_grasp(candidates)
        if best is None:
            print("파지 가능한 후보를 찾지 못했습니다.")
            return False

        mass_kg = estimate_mass_area_based(best.width_mm)
        target_force = self.planner.calculate_force(best.width_mm, mass_kg, f_external, mode=mode)

        # 접촉점(카메라 3D, mm) 두 점의 중점을 잡는 지점으로 사용, 핸드-아이 변환으로 로봇 좌표화
        mid_cam = (np.array(best.contact_points[0]) + np.array(best.contact_points[1])) / 2
        try:
            robot_xyz = camera_to_robot(mid_cam)
        except RuntimeError as e:
            print(f"[RobotSystem] {e} approach_pos를 그대로 사용합니다.")
            robot_xyz = None

        pick_pos = list(approach_pos)
        if robot_xyz is not None:
            pick_pos[0], pick_pos[1], pick_pos[2] = float(robot_xyz[0]), float(robot_xyz[1]), float(robot_xyz[2])
            # 자세(rx,ry,rz)는 캘리브레이션 대상이 아니므로 approach_pos 값을 그대로 유지

        return self.controller.execute_grasp(approach_pos, pick_pos, target_force)

## 8. 파지 실행 - RobotSystem 인스턴스 생성

In [ ]:
def move_fn(pos):
    movel(posx(pos), vel=SPEED_L, acc=ACC_L, ref=DR_BASE)
    wait(1)

def gripper_close_fn(current):
    gripper_close(current=current)
    wait(1)

def gripper_open_fn():
    gripper_open()
    wait(1)

system = RobotSystem(
    move_fn=move_fn,
    gripper_close_fn=gripper_close_fn,
    gripper_open_fn=gripper_open_fn,
    get_force_sensor_fn=None,  # F/T 센서 연동 시 함수 전달
)

print("RobotSystem 준비 완료.")

## 9. LUT 영구 저장/로드 (json)
커널을 껐다 켜도 `planner.lut`가 남아있도록 파일로 저장합니다. 파일 경로는 노트북과 같은 폴더의 `grasp_lut.json` 입니다.

In [ ]:
LUT_PATH = Path("grasp_lut.json")

def load_lut(planner: GraspPlanner):
    if LUT_PATH.exists():
        with open(LUT_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        planner.lut = {float(k): v for k, v in data.items()}
        print(f"LUT 로드 완료: {len(planner.lut)}개 항목")
    else:
        print("저장된 LUT 없음 - 빈 테이블로 시작")

def save_lut(planner: GraspPlanner):
    with open(LUT_PATH, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in planner.lut.items()}, f, indent=2, ensure_ascii=False)
    print(f"LUT 저장 완료: {len(planner.lut)}개 항목 → {LUT_PATH.resolve()}")

load_lut(system.planner)

## 10. 실험 루프 (배치검토형) - 여러 번 시도를 쌓아두고 나중에 한번에 검토
**사용법**
1. `try_force(width, force, approach_pos=PICK_POS)`를 여러 번 실행 (성공/실패 상관없이 계속 쌓임)
2. 다 시도한 뒤 `review_trials()`로 전체 목록을 표로 확인
3. `accept_trial(index)` / `reject_trial(index)`로 각 시도를 판정 (index는 review_trials() 표의 번호)
4. accept된 것만 LUT(`grasp_lut.json`)에 반영되고, 성공/실패 전체 이력은 `grasp_trials.json`에 남음

In [ ]:
TRIAL_LOG_PATH = Path("grasp_trials.json")
trial_log = []  # 이번 세션의 시도 목록: [{width, force, ok, accepted, note}, ...]

def load_trial_log():
    global trial_log
    if TRIAL_LOG_PATH.exists():
        with open(TRIAL_LOG_PATH, "r", encoding="utf-8") as f:
            trial_log = json.load(f)
        print(f"이전 실험 이력 로드: {len(trial_log)}건")
    else:
        print("이전 실험 이력 없음 - 새로 시작")

def save_trial_log():
    with open(TRIAL_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(trial_log, f, indent=2, ensure_ascii=False)

def try_force(width_mm: float, force: float, approach_pos, pick_pos=None, note: str = ""):
    """시도를 실행하고 trial_log에 append (판정은 아직 안 함, accepted=None)."""
    pick_pos = pick_pos or approach_pos
    ok = system.controller.execute_grasp(approach_pos, pick_pos, force)
    trial_log.append({
        "index": len(trial_log),
        "width": width_mm,
        "force": force,
        "ok": ok,          # 로봇 동작이 에러 없이 끝났는지 (파지 성공 여부와는 다름!)
        "accepted": None,  # True=성공(LUT 반영), False=실패, None=아직 미판정
        "note": note,
    })
    save_trial_log()
    i = len(trial_log) - 1
    print(f"[{i}] 시도 기록됨: width={width_mm}mm, force={force}N, ok={ok}")
    print("물체를 놓치지 않고 들었는지 확인하세요. 판정은 나중에 review_trials() 후 해도 됩니다.")
    return i

def review_trials():
    """전체 시도 목록을 표로 출력."""
    print(f"{'idx':>3} | {'width(mm)':>9} | {'force(N)':>8} | {'ok':>5} | {'accepted':>8} | note")
    for t in trial_log:
        print(f"{t['index']:>3} | {t['width']:>9} | {t['force']:>8} | {str(t['ok']):>5} | {str(t['accepted']):>8} | {t['note']}")

def accept_trial(index: int, note: str = ""):
    t = trial_log[index]
    t["accepted"] = True
    if note:
        t["note"] = note
    system.planner.add_calibration(t["width"], t["force"])
    save_lut(system.planner)
    save_trial_log()
    print(f"✅ [{index}] LUT 반영: width={t['width']}mm → force={t['force']}N")

def reject_trial(index: int, note: str = ""):
    t = trial_log[index]
    t["accepted"] = False
    if note:
        t["note"] = note
    save_trial_log()
    print(f"❌ [{index}] 기록만 남기고 LUT 반영 안 함: width={t['width']}mm, force={t['force']}N")

load_trial_log()

In [ ]:
# 예시:
# try_force(50, 12, approach_pos=PICK_POS) #50mm, 12N을 주었을 때
# try_force(50, 15, approach_pos=PICK_POS)
# try_force(50, 18, approach_pos=PICK_POS, note="살짝 흔들렸지만 안 떨어짐")
# review_trials() #표로 try_force를 진행한 결과들을 출력한다.
#
# 표 보고 나서 판정:
# accept_trial(2)   # 3번째 시도(index=2)를 최종 채택
# reject_trial(0)
# reject_trial(1)

## 11. 방정식 피팅 (fitted 모드) - accept된 실측 데이터로 physics 계수 보정
`accept_trial()`로 LUT에 쌓인 데이터는 지금까지 `naive` 모드에서만 쓰였습니다.
여기서는 그 데이터로 힘평형식(`F = k·M·g / 2μ`)의 비례상수를 역산해서, **처음 시도해보는 폭에도** 그 근사식으로 힘을 예측하게 합니다.

**사용법**
1. `try_force()` + `accept_trial()`로 최소 2개 이상 데이터를 쌓는다
2. `fit_physics_constant()` 실행 → `system.planner.fitted_C`에 저장됨
3. `system.run_task(..., mode="fitted")`로 새 물체에 적용

In [ ]:
def fit_physics_constant():
    """accept된 시도들로 F = C * mass 관계의 C를 최소자승법(원점 지나는 직선)으로 추정."""
    accepted = [t for t in trial_log if t["accepted"]]
    if len(accepted) < 2:
        print(f"accept된 데이터가 {len(accepted)}개뿐이라 회귀는 못 함 (2개 이상 필요).")
        return None

    widths = np.array([t["width"] for t in accepted])
    forces = np.array([t["force"] for t in accepted])
    masses = np.array([estimate_mass_area_based(w) for w in widths])

    # F = C * mass  (원점을 지나는 직선으로 피팅)
    C = float(np.sum(masses * forces) / np.sum(masses ** 2))
    print(f"피팅된 계수 C = {C:.3f}  (데이터 {len(accepted)}개 사용)")
    return C

system.planner.fitted_C = fit_physics_constant()

In [ ]:
# 세 가지 모드 비교 예시:
# system.run_task(color_image, depth_frame, approach_pos=PICK_POS, mode="naive")    # LUT에 정확히 같은 폭 있을 때만
# system.run_task(color_image, depth_frame, approach_pos=PICK_POS, mode="physics")  # mu=0.5, k=1.5 고정 가정값 사용
# system.run_task(color_image, depth_frame, approach_pos=PICK_POS, mode="fitted")   # 실측 데이터로 보정된 계수 사용
#
# 새로운 시도가 accept될 때마다 fit_physics_constant()를 다시 돌리면 fitted_C가 점점 정확해집니다.
# system.planner.fitted_C = fit_physics_constant()